# OWM — Colab GPU training

**Ishlatishdan oldin:** `Runtime → Change runtime type → T4 GPU` ni tanlang.

Bu notebook laptopdagi BIR XIL kodni ishga tushiradi. Hech narsa o'zgartirilmaydi —
faqat `device` avtomatik `cuda` bo'ladi va AMP yoqiladi.

| | Laptop | Colab T4 |
|---|---|---|
| epoch vaqti | ~14 s | ~0.5 s |
| batch size | 4 | 32–64 |
| sessiya | cheksiz | ~12 soat |

## 1. GPU tekshiruvi

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU yoqilmagan! Runtime -> Change runtime type -> T4 GPU"

## 2. Google Drive'ni ulash

Colab sessiyasi o'chganda hamma narsa yo'qoladi. Checkpointlar Drive'da saqlanadi —
shuning uchun uzilsa ham davom ettirish mumkin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/OWM')
(DRIVE_ROOT / 'models').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'experiments').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'data' / 'raw').mkdir(parents=True, exist_ok=True)

print('Drive tayyor:', DRIVE_ROOT)

## 3. Kodni GitHub'dan olish

⚠️ Quyidagi `GITHUB_URL` ni o'z repozitoriyangiz manziliga almashtiring.

In [ ]:
GITHUB_URL = 'https://github.com/ozodbek369/own-model.git'

import os, shutil

if os.path.exists('/content/owm'):
    shutil.rmtree('/content/owm')

!git clone -q $GITHUB_URL /content/owm

os.chdir('/content/owm')
!ls

## 4. Checkpoint va natijalarni Drive'ga bog'lash

Symlink: kod `models/` ga yozadi, aslida Drive'ga tushadi.

In [ ]:
for name in ['models', 'experiments']:
    local = Path('/content/owm') / name
    if local.is_symlink() or local.is_file():
        local.unlink()
    elif local.is_dir():
        shutil.rmtree(local)
    local.symlink_to(DRIVE_ROOT / name)
    print(f'{name} -> {DRIVE_ROOT / name}')

## 5. Ma'lumot

**CIFAR-10 uchun bu katakni o'tkazib yuborsangiz ham bo'ladi** -
torchvision uni o'zi yuklab oladi (Colab'da ~10 sekund).

Bu katak faqat o'z rasmlaringiz (`dataset: folder`) kerak bo'lganda ishlatiladi.
Ularni Drive'dagi `MyDrive/OWM/data/raw/` ga bir marta yuklab qo'ying.

In [ ]:
!mkdir -p /content/owm/data/raw
!cp -n "$DRIVE_ROOT/data/raw/"* /content/owm/data/raw/ 2>/dev/null || true

import subprocess
n = len(list(Path('/content/owm/data/raw').glob('*')))
print('Rasmlar soni:', n)

if n == 0:
    print('\n⚠️  Drive/OWM/data/raw bo\'sh. Rasmlarni o\'sha papkaga yuklang.')

## 6. Kutubxonalar

In [ ]:
# Colab'da torch/torchvision allaqachon bor — qolganini o'rnatamiz
!pip install -q matplotlib

## 7. Training - CIFAR-10, 60,000 rasm

| | Laptop | Colab T4 |
|---|---|---|
| epoch vaqti | ~6 daqiqa | ~15 sekund |
| 100 epoch | ~11 soat | **~25 daqiqa** |

Sessiya uzilsa: shu katakni qayta bosing. `resume: true` bo'lgani uchun
Drive'dagi oxirgi checkpoint'dan davom etadi - boshidan boshlamaydi.

In [ ]:
!python training/train_autoencoder.py --config configs/autoencoder_cifar.json --run-name ae_cifar_full --epochs 100 --batch-size 256 --num-workers 2 --save-every 10 --sample-every 10

## 8. Natijalar

In [ ]:
!python evaluation/plot_curves.py --run ae_cifar_full

import glob
from IPython.display import Image, display

display(Image('/content/owm/experiments/ae_cifar_full/loss_curve.png'))

# Namunalar: yuqori qator original, pastki qator model tiklagani
for p in sorted(glob.glob('/content/owm/experiments/ae_cifar_full/samples/*.png'))[-3:]:
    print(p.split('/')[-1])
    display(Image(p))


---

Checkpointlar `MyDrive/OWM/models/` da qoldi. Sessiya o'chsa ham yo'qolmaydi.